# Annotate merged single cells with metadata from platemap file

## Import libraries

In [1]:
import json
import os
import pathlib
import shutil
import sys
from collections import defaultdict

import duckdb
import natsort
import numpy as np
import pandas as pd
import polars as pl
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

In [ ]:
if in_notebook:
    import tqdm.notebook as tqdm

    plate_name = "plate_2"
else:
    import tqdm

    argparser = argparse.ArgumentParser()

    argparser.add_argument(
        "--plate_name",
        type=str,
        help="Name of the plate to analyze",
    )
    args = argparser.parse_args()
    plate_name = args.plate_name

In [2]:
def read_and_cast(path: pathlib.Path, conflicts: list[str]) -> pl.DataFrame:
    """
    Cast types for the dfs such that they are all similar

    Parameters
    ----------
    path : pathlib.Path
        path of the parquet file to read
    conflicts : list[str]
        Expected conflicting columns that should be cast to Float64. Only casts if the column is present in the df.

    Returns
    -------
    pl.DataFrame
        the read and casted dataframe
    """
    df = pl.read_parquet(path)
    cast_exprs = [
        pl.col(col).cast(pl.Float64) for col in conflicts if col in df.columns
    ]
    if cast_exprs:
        df = df.with_columns(cast_exprs)
    return df

## Set paths and variables

In [ ]:
image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(f"{root_dir}/data").resolve(strict=True)
image_base_dir = pathlib.Path(f"{image_base_dir}/processed_data/").resolve(strict=True)
converted_profiles_dir = pathlib.Path(
    f"{image_base_dir}/5.converted_profiles/{plate_name}"
).resolve(strict=True)

combined_profiles_path = pathlib.Path(
    f"{image_base_dir}/6.combined_profiles/{plate_name}"
).resolve()
combined_profiles_path.mkdir(exist_ok=True)

In [4]:
# well_fov_timepoints - get all the well_fov_timepoints that we have extracted features for
converted_profiles = [
    x for x in tqdm.tqdm(converted_profiles_dir.glob("*")) if x.is_dir()
]
# sort the converted profiles in natural order
converted_profiles = natsort.natsorted(converted_profiles)
# get all the converted profile parquet files as a list of paths to use downstream
converted_profiles = [
    list(x.glob("**/*.parquet"))[0]
    for x in tqdm.tqdm(
        converted_profiles,
        desc="Getting converted profiles",
        total=len(converted_profiles),
    )
    if len(list(x.glob("**/*.parquet"))) > 0
]
converted_profiles = natsort.natsorted(converted_profiles)
print(f"Number of converted profiles: {len(converted_profiles)}")

0it [00:00, ?it/s]

Getting converted profiles:   0%|          | 0/22848 [00:00<?, ?it/s]

Number of converted profiles: 22807


In [5]:
# check if any of the parquet files have buffer 0
# ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes
# find the files and print them
total = len(converted_profiles)
zero_size_files = 0
for parquet_file in tqdm.tqdm(converted_profiles):
    if parquet_file.stat().st_size == 0:
        zero_size_files += 1
        # # unlink the file
        # if parquet_file.is_file():
        #     parquet_file.unlink()
        # elif parquet_file.is_dir():
        #     shutil.rmtree(parquet_file)

print(f"Total files: {total}, Empty files: {zero_size_files}")

  0%|          | 0/22807 [00:00<?, ?it/s]

Total files: 22807, Empty files: 0


In [6]:
converted_profiles_df = pd.DataFrame({"converted_profile_path": converted_profiles})
converted_profiles_df["well_fov_time"] = converted_profiles_df[
    "converted_profile_path"
].apply(lambda x: x.parent.name)
converted_profiles_df["well_fov"] = converted_profiles_df[
    "converted_profile_path"
].apply(lambda x: ("_").join(x.parent.name.split("_")[:2]))
converted_profiles_df

,converted_profile_path,well_fov_time,well_fov
0,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,B2_1_T0001,B2_1
1,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,B2_1_T0002,B2_1
2,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,B2_1_T0003,B2_1
3,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,B2_1_T0004,B2_1
4,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,B2_1_T0005,B2_1
...,...,...,...
22802,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,O5_4_T0098,O5_4
22803,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,O5_4_T0099,O5_4
22804,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,O5_4_T0100,O5_4
22805,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,O5_4_T0101,O5_4


In [7]:
converted_profiles_df = converted_profiles_df.loc[
    converted_profiles_df["well_fov"] == "C2_3"
]
converted_profiles_df

,converted_profile_path,well_fov_time,well_fov
1835,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0002,C2_3
1836,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0003,C2_3
1837,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0004,C2_3
1838,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0005,C2_3
1839,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0006,C2_3
...,...,...,...
1931,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0098,C2_3
1932,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0099,C2_3
1933,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0100,C2_3
1934,/home/lippincm/4TB_A/pyroptosis_live-cell_time...,C2_3_T0101,C2_3


In [11]:
converted_profiles_df.loc[converted_profiles_df["well_fov_time"] == "C2_3_T0001"]

,converted_profile_path,well_fov_time,well_fov


In [8]:
# loop through unique well fovs then combine all timepoints for each well fov and save as a single parquet file
for well_fov, group in tqdm.tqdm(
    converted_profiles_df.groupby("well_fov"),
    desc="Combining profiles",
    total=converted_profiles_df["well_fov"].nunique(),
):
    well_fov_path = pathlib.Path(f"{combined_profiles_path}/{well_fov}.parquet")
    well_fov_path.parent.mkdir(parents=True, exist_ok=True)
    if well_fov_path.exists():
        continue
    # if not well_fov_path.exists():
    # get the converted profile paths for the current well fov time
    tmp_df = converted_profiles_df[converted_profiles_df["well_fov"] == well_fov]

    # read each parquet file and concatenate them into a single dataframe
    combined_well_fov_paths = tmp_df["converted_profile_path"].tolist()
    combined_well_fov_paths = [str(x) for x in combined_well_fov_paths]

    # Step 1: find all conflicting columns
    schemas = [pl.read_parquet_schema(p) for p in combined_well_fov_paths]

    col_types = defaultdict(set)
    for schema in schemas:
        for col, dtype in schema.items():
            col_types[col].add(dtype)

    conflicts = {col: types for col, types in col_types.items() if len(types) > 1}
    if conflicts:
        print(f"Conflicting columns found in {well_fov}:")
        for col, types in conflicts.items():
            print(f"  Column: {col}, Types: {types}")

    # Step 2: read each file, casting conflicts to Float64 before concat

    frames = [
        read_and_cast(p, conflicts)
        for p in tqdm.tqdm(
            combined_well_fov_paths, leave=False, desc="Reading and casting files"
        )
    ]
    combined_df = pl.concat(
        frames, how="diagonal"
    )  # diagonal handles missing columns too
    # fix mixed types in object columns before saving to parquet
    # for col in combined_df.select_dtypes(include="object").columns:
    #     combined_df[col] = combined_df[col].astype(str)
    # for col in combined_df.columns:
    #     if col == "Metadata_Cells_Number_Object_Number":
    #         combined_df[col] = combined_df[col].astype(int)
    # save the combined dataframe as a parquet file
    combined_df.write_parquet(well_fov_path)

Combining profiles:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
combined_df = pd.read_parquet(well_fov_path)

In [10]:
combined_df

,Metadata_Well,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_Nuclei_Number_Object_Number,...,Nuclei_Texture_Variance_CL640_3_03_256,Nuclei_Texture_Variance_NucleoLive_3_00_256,Nuclei_Texture_Variance_NucleoLive_3_01_256,Nuclei_Texture_Variance_NucleoLive_3_02_256,Nuclei_Texture_Variance_NucleoLive_3_03_256,Nuclei_Texture_Variance_SYTOXGreen_3_00_256,Nuclei_Texture_Variance_SYTOXGreen_3_01_256,Nuclei_Texture_Variance_SYTOXGreen_3_02_256,Nuclei_Texture_Variance_SYTOXGreen_3_03_256,Metadata_single_cell_count
0,C2,1,C2_3,3,C2_3_1,1,1.0,1,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
1,C2,1,C2_3,3,C2_3_1,1,2.0,2,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
2,C2,1,C2_3,3,C2_3_1,1,3.0,3,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
3,C2,1,C2_3,3,C2_3_1,1,4.0,4,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
4,C2,1,C2_3,3,C2_3_1,1,5.0,5,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155627,C2,102,C2_3,3,C2_3_102,1,1830.0,1830,1830,1830.0,...,10.040000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1834
155628,C2,102,C2_3,3,C2_3_102,1,1831.0,1831,1831,1831.0,...,5.375000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1834
155629,C2,102,C2_3,3,C2_3_102,1,1832.0,1832,1832,1832.0,...,9.734375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1834
155630,C2,102,C2_3,3,C2_3_102,1,1833.0,1833,1833,1833.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1834
